# Notebook 02 — Data Preprocessing & Feature Engineering
**Credit Card Approval Prediction System**

This notebook covers:
- Duplicate removal
- Missing value analysis and handling
- `data_cleaning()` function walkthrough
- Binary target label engineering via `is_risky()`
- Dataset merge

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
app    = pd.read_csv('../data/application_record.csv')
credit = pd.read_csv('../data/credit_record.csv')
print(f'Shapes — app: {app.shape}, credit: {credit.shape}')

## 2. Step 3.1 — Remove Duplicates

In [ ]:
subset_cols = [
    'CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY','CNT_CHILDREN',
    'AMT_INCOME_TOTAL','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','DAYS_BIRTH','DAYS_EMPLOYED',
    'FLAG_MOBIL','FLAG_WORK_PHONE','FLAG_PHONE','FLAG_EMAIL',
    'OCCUPATION_TYPE','CNT_FAM_MEMBERS'
]

before = len(app)
app.drop_duplicates(subset=subset_cols, keep='first', inplace=True)
print(f'Rows before: {before} | Rows after: {len(app)} | Removed: {before - len(app)}')

## 3. Step 3.2 — Handle Missing Values

In [ ]:
print('=== Missing value counts ===')
print(app.isnull().sum())
print('\n=== Missing value proportions ===')
print(app.isnull().mean().round(4))

In [ ]:
# OCCUPATION_TYPE has ~30.5% missing → drop the column
app.drop(columns=['OCCUPATION_TYPE'], inplace=True, errors='ignore')
print(f'OCCUPATION_TYPE dropped. Remaining columns: {len(app.columns)}')
print('Remaining missing values:', app.isnull().sum().sum())

## 4. Step 3.3 — Feature Engineering

In [ ]:
# Fix negative DAYS_BIRTH and DAYS_EMPLOYED
app['DAYS_BIRTH']    = app['DAYS_BIRTH'].abs()
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].abs()

print('DAYS_BIRTH sample:', app['DAYS_BIRTH'].head().values)
print('DAYS_EMPLOYED sample:', app['DAYS_EMPLOYED'].head().values)

In [ ]:
# Family dependency ratio
app['family_dependency'] = app['CNT_CHILDREN'] / app['CNT_FAM_MEMBERS'].replace(0, 1)
print('family_dependency stats:')
print(app['family_dependency'].describe())

In [ ]:
# Categorical mapping
HOUSING_MAP   = {'House / apartment':0,'Rented apartment':1,'With parents':2,
                 'Municipal apartment':3,'Co-op apartment':4,'Office apartment':5}
INCOME_MAP    = {'Working':0,'Commercial associate':1,'Pensioner':2,'State servant':3,'Student':4}
EDUCATION_MAP = {'Higher education':0,'Secondary / secondary special':1,
                 'Incomplete higher':2,'Lower secondary':3,'Academic degree':4}
FAMILY_MAP    = {'Married':0,'Single / not married':1,'Civil marriage':2,'Separated':3,'Widow':4}

app['NAME_HOUSING_TYPE']   = app['NAME_HOUSING_TYPE'].map(HOUSING_MAP).fillna(0)
app['NAME_INCOME_TYPE']    = app['NAME_INCOME_TYPE'].map(INCOME_MAP).fillna(0)
app['NAME_EDUCATION_TYPE'] = app['NAME_EDUCATION_TYPE'].map(EDUCATION_MAP).fillna(0)
app['NAME_FAMILY_STATUS']  = app['NAME_FAMILY_STATUS'].map(FAMILY_MAP).fillna(0)

print('Mapping complete. Sample:')
app[['NAME_HOUSING_TYPE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS']].head()

## 5. Step 3.4 — Binary Target Label (is_risky)

In [ ]:
def is_risky(status):
    """
    STATUS codes:
      0 = 1-29 days overdue    → low risk
      1 = 30-59 days overdue   → medium risk
      2 = 60-89 days overdue   → HIGH
      3 = 90-119 days overdue  → HIGH
      4 = 120-149 days overdue → HIGH
      5 = bad debt/write-off   → HIGH
      C = paid off that month  → good
      X = no loan that month   → neutral
    Returns 1 (risky) or 0 (good).
    """
    return 1 if str(status) in ['2','3','4','5'] else 0

credit['risk'] = credit['STATUS'].apply(is_risky)
print('Risk distribution:')
print(credit['risk'].value_counts())

## 6. Step 3.5 — Aggregate Credit Records & Merge

In [ ]:
agg = credit.groupby('ID').agg(
    open_month  = ('MONTHS_BALANCE', 'min'),
    end_month   = ('MONTHS_BALANCE', 'max'),
    total_loans = ('MONTHS_BALANCE', 'count'),
    emi_paid_off= ('STATUS', lambda s: (s == 'C').sum()),
    emi_overdue = ('STATUS', lambda s: s.isin(['1','2','3','4','5']).sum()),
    target      = ('risk', 'max'),
).reset_index()

agg['window'] = agg['end_month'] - agg['open_month']

print(f'Aggregated credit shape: {agg.shape}')
print('Target distribution:')
print(agg['target'].value_counts())
agg.head()

In [ ]:
merged = pd.merge(app, agg, on='ID', how='inner')
print(f'Merged dataset shape: {merged.shape}')
print(f'Target distribution in merged set:')
print(merged['target'].value_counts())
print(f'  Approved (0): {(merged.target==0).sum()} | Rejected (1): {(merged.target==1).sum()}')
merged.head()

In [ ]:
# Visualize target imbalance
fig, ax = plt.subplots(figsize=(6, 4))
merged['target'].value_counts().plot.bar(
    ax=ax, color=['#10b981','#ef4444'], edgecolor='none', rot=0)
ax.set_xticklabels(['Approved (0)', 'Rejected (1)'])
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()